# ALBEDO: simulate a source

Choose the source and the options in the **Parameters** cell, then run all cells
(*Run All*). The simulation can be saved as CSV and compared with a measured
spectrum described in `configs/measurements/`.

In [ ]:
import matplotlib.pyplot as plt

from albedo import load_source, plotting, simulate
from albedo.experiment import ROOT, compare, process_measurement
from albedo.figures import plot_comparison, plot_simulation

plotting.use_style()
RESULTS = ROOT / "results"

## Parameters

In [ ]:
SOURCE = "Cs-137"            # Cs-137, Ba-133, Mn-54, Y-88 (any file in data/sources)
SETUP = "csi_747E"           # file in configs/setups
N_HISTORIES = 3e7            # ~1e6 histories per second
SEED = 1

SAVE_CSV = True              # write the simulated spectrum to results/
SAVE_FIGURE = False          # write the figure to figures/

COMPARE = True               # compare with a measured spectrum
MEASUREMENT = "cs137_2025-08-23"   # file in configs/measurements
NORMALIZATION = "absolute"   # "absolute" (source activity) or "window"

LOG = False                  # logarithmic vertical axis
COMPONENTS = False           # split the simulation by origin (direct, Cu, Sn, Pb)
BIN_WIDTH = 2.0              # keV, only used when not comparing

## Source

In [ ]:
source = load_source(SOURCE)
print(source)

## Simulation

In [ ]:
result = simulate(SETUP, source, N_HISTORIES, seed=SEED)

print(f"{result.n_histories:.3e} histories, {len(result.deposited)} detected "
      f"(efficiency {result.efficiency:.3e}), equivalent to {result.decays:.3e} decays")
for name, mask in result.components().items():
    print(f"  {name:12s} {mask.sum():9d}  ({mask.mean():6.1%})")

if SAVE_CSV:
    path = result.to_csv(RESULTS / f"sim_{SOURCE}_{SETUP}.csv", bin_width=BIN_WIDTH, seed=SEED)
    print("saved", path)

## Spectrum

In [ ]:
if COMPARE:
    measured = process_measurement(MEASUREMENT)
    if measured.source != SOURCE:
        raise ValueError(f"{MEASUREMENT} was measured with {measured.source}, not {SOURCE}")
    comparison = compare(measured, result, NORMALIZATION)
    fig, _ = plot_comparison(comparison, components=COMPONENTS, log=LOG)

    e = measured.energy
    print(f"{NORMALIZATION} normalization, scale = {comparison.scale:.4g}")
    for lo, hi in [(40, 150), (150, 300), (300, 470), (470, 560), (600, 720)]:
        sel = (e >= lo) & (e < hi)
        ratio = measured.net[sel].sum() / comparison.simulated[sel].sum()
        print(f"  {lo:4d}-{hi:4d} keV   measured/simulated = {ratio:.3f}")
    if SAVE_CSV:
        print("saved", comparison.to_csv(RESULTS / f"compare_{MEASUREMENT}.csv"))
    name = f"compare_{MEASUREMENT}"
else:
    fig, _ = plot_simulation(result, bin_width=BIN_WIDTH, components=COMPONENTS, log=LOG, seed=SEED)
    name = f"sim_{SOURCE}_{SETUP}"

if SAVE_FIGURE:
    plotting.save(fig, name)
plt.show()